# The nine Cohere models on SGLang, measured

This notebook measures the SGLang column of `docs/model_backends.md` (#52). Until now that column was read off SGLang's model registry, not run.

Each model goes through one real SGLang server, the way `api:` users run it, and gets four checks:

| Check | What it records |
|---|---|
| **registry** | Whether this SGLang's source declares the model's architecture. This is the reading the doc has today. |
| **load** | Whether the server comes up. If it doesn't, the reason is classified: *no implementation*, *transformers too old*, *out of memory*, *download or gating*, or *other*, with the log line that says so. |
| **text** | The same five-language probes and pass rule as the `hf:` column (`colab_cohere_models.ipynb`): PASS at 4/5 correct or better, PARTIAL for 1–3, FAIL for 0. |
| **image** | Vision architectures only, if they load: a picture of a red square, asked "what colour is it?" through `Sample.images` (#50). |

**What it fixes along the way** (each one is recorded in the results, never silently):
- **SGLang 0.5.20 bug:** a multimodal model on SGLang's Transformers backend (Aya Vision) crashes on its first request with `UnifiedRadixCache.dec_lock_ref() missing 1 required positional argument: 'params'`. The bug is still on SGLang `main`. `PATCH_SGLANG` gives that argument its missing default. The one code path that omits it (radix cache disabled) returns before reading it.
- **North:** SGLang pins transformers 5.12.1, which can't read `cohere_compass`. A model that fails for that reason is retried in a **second** SGLang venv with transformers `ALT_TRANSFORMERS`, served through SGLang's Transformers backend, and labelled as a non-default configuration. That backend then rejects the `embedding_rowwise` TP style, which transformers ≥ 5.15 adds for tied embeddings. That's a second SGLang bug, and a third follows: North's `rope_parameters` has `None` for its full-attention layers (they use no RoPE), which SGLang's torch.compile check can't handle. `PATCH_SGLANG` (`compat.py --patch-sglang`) fixes both. SGLang doesn't recognise North's nested `mrope_section`, so it passes plain positions: exact for text (North expands them to all axes, as HF does for text tokens), approximate for image tokens.
- **Tiny Aya's score gap against `hf:`:** for the models in `DIAGNOSE`, the same five prompts go through three paths on this GPU: SGLang's chat API, SGLang's `/generate` fed `hf:`'s exact token ids, and `hf:` itself. The prompt-token counts are compared too (an extra BOS would show). The verdict says where the difference comes from.

**Any GPU:** SGLang runs with the machine's own CUDA toolkit, the setup that worked on the A100 runs. The models run in their own bf16. AuditKIT's `compat.py --sglang-launch` adds only what the GPU needs:
- **Hopper / Blackwell:** DeepGEMM is switched off (`SGLANG_ENABLE_JIT_DEEPGEMM=0`). It JIT-compiles at startup with nvcc ≥ 12.9 and failed every load on the RTX PRO 6000; it serves FP8 GEMMs only, so bf16 loses nothing.
- **T4 / V100 (no bf16):** fp16.
- **RTX PRO 6000 / RTX 50-series (SM 12.x) with a CUDA toolkit older than 12.9:** **Triton attention and PyTorch sampling** from the start. FlashInfer 0.6.18 compiles for SM 12.x only with CUDA ≥ 12.9. With an older toolkit it fails every load with the misleading "FlashInfer requires GPUs with sm75 or higher".
- **If FlashInfer's kernels still don't build on this machine,** that model is relaunched with Triton attention and PyTorch sampling. Neither needs a CUDA compiler. Every later model then starts on Triton directly, and the results label it (`TRITON_FALLBACK`).

The advice is recorded in the results.

**Runtime:** anything from a T4 up. Models larger than the GPU are SKIP. Use **80 GB or more** to run all nine in bf16, as the `hf:` column was. On a 40 GB A100, the two 32B models are SKIP, unless you set `ALLOW_FP8_FOR_LARGE = True`, which serves them in FP8. That result is labelled and is not comparable to the bf16 column. Each model's weights are deleted after testing (`DELETE_AFTER`).

**Tokens:** the notebook asks for them in hidden boxes, and they are never saved in the notebook.
- `GITHUB_TOKEN`: installs AuditKIT.
- `HF_TOKEN`: all nine models are gated, so accept their licences on the Hub first.

**Time:** about 2 to 3 hours, mostly downloads.

In [ ]:
# ---- settings ----
BRANCH = "main"  # launch advice and the five SGLang patches are merged (#56)
REPO_URL = "github.com/aryapratinavseth/AuditKIT-Internal.git"
SGLANG = "sglang>=0.5.18,<0.6"
PORT = 30000
DELETE_AFTER = True             # free the disk after each model: the nine don't fit on Colab together
LOAD_TIMEOUT_S = 3600           # includes the download: a 32B model is ~65 GB
PATCH_SGLANG = True             # five SGLang bugs (see above: Aya Vision, North, Triton sliding windows); recorded in VERSIONS
ALT_TRANSFORMERS = "5.16.1"     # retry venv for models SGLang's transformers pin can't read (North); None = off
ALLOW_FP8_FOR_LARGE = False     # 32B on a <80 GB GPU in FP8: labelled, NOT comparable to the bf16 hf: column
TRITON_FALLBACK = True          # FlashInfer's kernels didn't build here -> Triton attention + PyTorch sampling (no nvcc)
DIAGNOSE = ["CohereLabs/tiny-aya-global"]   # three-way prompt comparison (see above)
# (repo, architecture, min GPU GB in bf16, is_vision_arch)
MODELS = [
    ("CohereLabs/tiny-aya-global", "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-fire",   "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-water",  "cohere2", 8,  False),
    ("CohereLabs/tiny-aya-earth",  "cohere2", 8,  False),
    ("CohereLabs/North-Micro-Vision-Instruct", "cohere_compass", 6, True),
    ("CohereLabs/aya-expanse-8b",  "cohere", 18, False),
    ("CohereLabs/aya-vision-8b",   "aya_vision", 20, True),
    ("CohereLabs/aya-expanse-32b", "cohere", 70, False),
    ("CohereLabs/aya-vision-32b",  "aya_vision", 75, True),
]

# `expect` is any one acceptable answer. Asking is not enough: a model that loads
# and answers in English scores the same as one that understood the question.
PROBES = [
    ("fr", "Quel est le plus grand port de France ? Un mot.",
     ("Marseille", "marseille", "Marseill")),
    ("hi", "\u092d\u093e\u0930\u0924 \u0915\u0940 \u0930\u093e\u091c\u0927\u093e\u0928\u0940 \u0915\u094d\u092f\u093e \u0939\u0948? \u090f\u0915 \u0936\u092c\u094d\u0926\u0964",
     ("\u0926\u093f\u0932\u094d\u0932\u0940", "\u0928\u090f\u0940 \u0926\u093f\u0932\u094d\u0932\u0940", "delhi")),
    ("zh", "\u4e2d\u56fd\u7684\u9996\u90fd\u662f\u54ea\u91cc\uff1f\u7528\u4e00\u4e2a\u8bcd\u3002",
     ("\u5317\u4eac", "beijing")),
    ("ar", "\u0645\u0627 \u0647\u064a \u0639\u0627\u0635\u0645\u0629 \u0645\u0635\u0631\uff1f \u0643\u0644\u0645\u0629 \u0648\u0627\u062d\u062f\u0629.",
     ("\u0627\u0644\u0642\u0627\u0647\u0631\u0629", "cairo")),
    ("th", "\u0e40\u0e21\u0e37\u0e2d\u0e07\u0e2b\u0e25\u0e27\u0e07\u0e02\u0e2d\u0e07\u0e1b\u0e23\u0e30\u0e40\u0e17\u0e28\u0e44\u0e17\u0e22\u0e04\u0e37\u0e2d\u0e2d\u0e30\u0e32\u0e2b\u0e48\u0e2d\u0e04\u0e23\u0e31\u0e1a? \u0e2b\u0e19\u0e34\u0e48\u0e07\u0e04\u0e33",
     ("\u0e01\u0e23\u0e38\u0e07\u0e40\u0e17\u0e1e", "bangkok")),
]
# "กทม." is the standard abbreviation of Bangkok: a correct answer the #49 probe list misses
PROBES = [(c, p, tuple(e) + (("กทม",) if c == "th" else ())) for c, p, e in PROBES]
RESULTS = []

In [ ]:
# ---- tokens: paste into the boxes (hidden input; never saved in the notebook) ----
import os
from getpass import getpass
for key, why in (("GITHUB_TOKEN", "required: installs AuditKIT"),
                 ("HF_TOKEN", "required: the Cohere models are gated")):
    if os.environ.get(key):
        print(f"{key}: already set in this runtime"); continue
    value = getpass(f"Paste {key} ({why}): ").strip()
    if value:
        os.environ[key] = value
print({k: bool(os.environ.get(k)) for k in ("GITHUB_TOKEN", "HF_TOKEN")})

In [ ]:
# ---- AuditKIT (client side, requests only) and a separate Python 3.12 venv for SGLang ----
import json, subprocess, sys
def sh(*a, **kw):
    return subprocess.run(a, capture_output=True, text=True, **kw)

REPO = "/content/AuditKIT-Internal"
url = f"https://{os.environ['GITHUB_TOKEN']}@{REPO_URL}"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, url, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", url, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[requests]"], check=True)
sys.path.insert(0, f"{REPO}/src")

# SGLang never shares an environment with vLLM/transformers pins (docs/SGLANG.md), and its
# outlines_core pin has no cp313 wheel: build its own 3.12 venv with uv, plus ninja for kernels.
def build_sglang_env(path, extra=()):
    # a Python 3.12 venv with SGLang (+ extra pins installed after it); returns its python
    py = f"{path}/bin/python"
    # a fresh runtime has no venv yet: never exec a python that isn't there
    if not (os.path.exists(py) and sh(py, "-c", "import sys; print(sys.version_info[:2] == (3, 12))").stdout.strip() == "True"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", "--seed", "--clear", path], check=True)
    for spec in (["--upgrade", "pip"], [SGLANG], ["ninja"], *[[e] for e in extra]):
        subprocess.run([py, "-m", "pip", "install", "-q", *spec], check=True)
    return py

def env_vars(py):
    env = {**os.environ, "PATH": f"{os.path.dirname(py)}:" + os.environ.get("PATH", "")}
    if os.path.isdir("/usr/local/cuda"):
        env.setdefault("CUDA_HOME", "/usr/local/cuda")
    env.pop("VLLM_ENABLE_V1_MULTIPROCESSING", None)
    env.update(launch_advice(py)["env"])            # this machine's needs (compat.py --sglang-launch)
    return env

_ADVICE = {}
def launch_advice(py):
    # what an SGLang server needs on THIS machine, whatever the GPU: AuditKIT's own compat.py
    # (#54), run inside the SGLang venv; an inline fallback for a branch without --sglang-launch
    if py not in _ADVICE:
        env = {**os.environ, "PATH": f"{os.path.dirname(py)}:" + os.environ.get("PATH", "")}
        if os.path.isdir("/usr/local/cuda"):
            env.setdefault("CUDA_HOME", "/usr/local/cuda")
        out = subprocess.run([py, f"{REPO}/src/auditkit/compat.py", "--sglang-launch", "x"], capture_output=True,
                             text=True, env=env).stdout
        line = next((l for l in out.splitlines() if l.startswith("sglang launch advice:")), None)
        if line:
            _ADVICE[py] = {"source": "compat.py", **json.loads(line.split(":", 1)[1])}
        else:
            import torch
            cc = torch.cuda.get_device_capability(0)
            _ADVICE[py] = {"source": "inline", "notes": [],
                           "env": {"SGLANG_ENABLE_JIT_DEEPGEMM": "0"} if cc[0] >= 9 else {},
                           "args": [] if cc[0] >= 8 else ["--dtype", "float16"]}
        print(f"  SGLang launch advice ({_ADVICE[py]['source']}): env={_ADVICE[py]['env']} "
              f"args={_ADVICE[py]['args']} notes={_ADVICE[py].get('notes')}", flush=True)
    return _ADVICE[py]

# The two SGLang fixes, inline, for a branch whose compat.py predates --patch-sglang:
# (i) UnifiedRadixCache.dec_lock_ref(): default `params` to None. The scheduler calls it without params
#     when the radix cache is disabled (multimodal models on the Transformers backend: Aya Vision); that
#     path returns before reading it.
# (j) "embedding_rowwise" -> "replicate" in the Transformers backend's TP-style table. transformers>=5.15
#     adds it for tied-embedding models (North); the backend reads the plan for nn.Linear only.
# All five are upstream bugs, unfixed on SGLang main as of 2026-09-30.
_INLINE_FIXES = (
    ("dec_lock_ref", "srt/mem_cache/unified_radix_cache.py",
     "        node_id: NodeId,\n        params: DecLockRefParams,\n        skip_swa: bool = False,\n",
     "        node_id: NodeId,\n        params: DecLockRefParams = None,\n        skip_swa: bool = False,\n"),
    ("embedding_rowwise", "srt/models/transformers.py",
     '        "moe_tp_experts": "replicate",\n    }.get(style, style)',
     '        "moe_tp_experts": "replicate",\n        "embedding_rowwise": "replicate",\n    }.get(style, style)'),
    # (l) Cohere2ForCausalLM never reports its sliding window: the Triton backend then gets kv_indptr=None
    ("cohere2_sliding_window", "srt/models/commandr.py",
     "class Cohere2ForCausalLM(CohereForCausalLM):\n    pass\n",
     "class Cohere2ForCausalLM(CohereForCausalLM):\n    def get_attention_sliding_window_size(self):\n"
     "        return getattr(self.config, 'sliding_window', None)\n"),
    # (m) the same gap in the Transformers backend (Aya Vision): Triton decode gets kv_indptr=None
    ("transformers_sliding_window", "srt/models/transformers.py",
     "        return loader.load_weights(weights, mapper=self.weight_mapper)\n\n\nclass CausalMixin:",
     "        return loader.load_weights(weights, mapper=self.weight_mapper)\n\n"
     "    def get_attention_sliding_window_size(self):\n"
     "        layer_types = getattr(self.text_config, 'layer_types', None) or getattr(self.config, 'layer_types', None)\n"
     "        if not layer_types or 'sliding_attention' not in layer_types:\n"
     "            return None\n"
     "        return getattr(self.text_config, 'sliding_window', None) or getattr(self.config, 'sliding_window', None)\n"
     "\n\nclass CausalMixin:"),
    # (n) North's rope_parameters has None for its no-RoPE full-attention layers; SGLang's
    #     torch.compile check calls .get on it ("'NoneType' object has no attribute 'get'")
    ("rope_parameters_none", "srt/models/transformers.py",
     '                rp.get("rope_type") == "dynamic" for rp in rope_params.values()\n',
     '                isinstance(rp, dict) and rp.get("rope_type") == "dynamic"\n'
     '                for rp in rope_params.values()\n'),
)

def patch_sglang_inline(py):
    site = sh(py, "-c", "import sglang, os; print(os.path.dirname(sglang.__file__))").stdout.strip()
    out = []
    for name, rel, old, new in _INLINE_FIXES:
        f = f"{site}/{rel}"
        src = open(f).read()
        if new in src:
            out.append(f"{name}: already patched")
        elif src.count(old) != 1:
            out.append(f"{name}: not applied (changed in this SGLang)")
        else:
            open(f, "w").write(src.replace(old, new)); out.append(f"{name}: patched")
    return "; ".join(out)

SGL_PY = build_sglang_env("/content/sglang-env")
SGL_ENVVARS = env_vars(SGL_PY)
def patch_sglang(py):
    # AuditKIT's own fix first (#54: compat.py --patch-sglang), so this run tests the code under review;
    # the inline copy above covers a branch that predates the flag
    out = sh(py, f"{REPO}/src/auditkit/compat.py", "--patch-sglang").stdout
    line = next((l for l in out.splitlines() if l.startswith("sglang patches:")), None)
    if line is None:
        return "inline: " + patch_sglang_inline(py)
    return "compat.py: " + line.split(":", 1)[1].strip()

SGL_PATCH = patch_sglang(SGL_PY) if PATCH_SGLANG else "off"
print("SGLang dec_lock_ref patch:", SGL_PATCH)
ALT_PY = None     # the transformers-upgraded venv, built only if a model needs it

import auditkit as ak, torch
VERSIONS = {"auditkit": ak.__version__,
            "commit": sh("git", "-C", REPO, "log", "--oneline", "-1").stdout.strip(),
            **{p: sh(SGL_PY, "-c", f"import importlib.metadata as m; print(m.version('{p}'))").stdout.strip() or None
               for p in ("sglang", "transformers", "torch", "flashinfer-python")},
            "sglang_patch": SGL_PATCH, "launch_advice": launch_advice(SGL_PY)}
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(VERSIONS, "|", GPU_NAME, f"{GPU_GB:.0f} GB")
print(sh(SGL_PY, f"{REPO}/src/auditkit/compat.py", "--cuda").stdout)

## Helpers

- **registry:** reads each model's `config.json` for its architecture class, then looks for that class name in SGLang's `srt/models/` source.
- **load:** starts the server, waits until it is healthy, and on failure classifies the log.
- **probes:** the `hf:` column's probes, sent through `api:`.

In [ ]:
import glob, json, re, shutil, signal, time
import requests
from huggingface_hub import hf_hub_download, scan_cache_dir
from auditkit.model import Request
from auditkit.model.api_gen import APIModel

def record(model, check, outcome, detail=""):
    RESULTS.append({"model": model, "check": check, "outcome": outcome, "detail": str(detail)[:600]})
    print(f"  [{outcome:8}] {model.split('/')[-1]:28s} {check:9s} {str(detail)[:150]}", flush=True)

SGL_SITE = sh(SGL_PY, "-c", "import sglang, os; print(os.path.dirname(sglang.__file__))").stdout.strip()

def registry(repo):
    # (architecture classes, the ones SGLang's source defines, and in which file)
    cfg = json.load(open(hf_hub_download(repo, "config.json", token=os.environ.get("HF_TOKEN"))))
    archs, found = cfg.get("architectures") or [], {}
    for path in glob.glob(f"{SGL_SITE}/srt/models/*.py"):
        text = open(path, encoding="utf-8", errors="ignore").read()
        for a in archs:
            if re.search(rf"\bclass\s+{a}\b|\b{a}\s*=|EntryClass[^\n]*\b{a}\b", text):
                found.setdefault(a, os.path.basename(path))
    return archs, found, cfg.get("model_type")

CAUSES = [  # first match wins; each is a regex over the server log
    ("kernel build", r"(Ninja build failed|ninja: build stopped|ptxas fatal|nvcc fatal|cannot find -l\w+|collect2: error|"
                     r"FlashInfer requires GPUs with sm|SM 12\.x requires CUDA)"),
    ("no implementation", r"(Model architectures? .* (are|is) not supported|not supported for now|"
                          r"Unsupported architecture|No model architecture|ModelRegistry)"),
    ("transformers too old", r"(does not recognize this architecture|model type `?\w+`? but Transformers does not)"),
    ("out of memory", r"(out of memory|OutOfMemory|Not enough memory|CUDA error: out of memory)"),
    ("download or gating", r"(GatedRepoError|401 Client Error|403 Client Error|RepositoryNotFound|not a valid model identifier)"),
    ("disk full", r"(No space left on device|os error 28)"),
]

def classify(log_text):
    # the server's final error decides, so a warning logged on the way (FlashInfer's "Failed to get
    # device capability ...") is never reported as the cause; the whole log only if there is none
    errors = [l for l in log_text.splitlines() if re.search(r"(Error|Exception)\b|error:|fatal", l)]
    if errors:
        last = errors[-1].strip()
        for cause, pattern in CAUSES:
            if re.search(pattern, last):
                return cause, last
        return "other", last
    for cause, pattern in CAUSES:
        m = re.search(pattern, log_text)
        if m:
            line = next((l for l in log_text.splitlines() if m.group(0) in l), m.group(0))
            return cause, line.strip()
    return "other", (log_text.strip().splitlines() or ["empty log"])[-1].strip()

def free_gb():
    out = sh("nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits").stdout.strip()
    return float(out.splitlines()[0]) / 1024 if out else 0.0

TRITON_ARGS = ("--attention-backend", "triton", "--sampling-backend", "pytorch")
TRITON_LABEL = " [non-default: Triton attention, FlashInfer's kernels didn't build on this machine]"
USE_TRITON = False              # sticky: once FlashInfer fails to build here, every later model skips it

def start(repo, log, py=None, extra=()):
    py = py or SGL_PY
    extra = (*extra, *(TRITON_ARGS if USE_TRITON else ()))
    env = env_vars(py)
    if "fp8" in extra:
        env.pop("SGLANG_ENABLE_JIT_DEEPGEMM", None)   # FP8 is what DeepGEMM is for
    args = [a for a in launch_advice(py)["args"] if a not in extra]
    return subprocess.Popen([py, "-m", "sglang.launch_server", "--model-path", repo, "--host", "127.0.0.1",
                             "--port", str(PORT), "--context-length", "4096", "--trust-remote-code", *args, *extra],
                            stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True, env=env)

def try_load(repo, log, extra, label):
    # launch, and on a failure that another configuration can get past, retry with it: Triton
    # attention for a kernel build failure, the second venv for a transformers too old for the model
    global USE_TRITON
    py, tried = SGL_PY, set()
    if USE_TRITON:
        label += TRITON_LABEL
    elif "triton" in launch_advice(py)["args"]:
        label += " [non-default: Triton attention, advised: FlashInfer can't build for this GPU with this CUDA toolkit]"
    while True:
        t0, proc = time.time(), start(repo, log, py=py, extra=extra)
        if wait(proc, LOAD_TIMEOUT_S):
            return proc, t0, label
        text = open(log, encoding="utf-8", errors="ignore").read()
        cause, line = classify(text) if proc.poll() is not None else ("timeout", f"not healthy after {LOAD_TIMEOUT_S}s")
        stop(proc)
        if cause == "kernel build" and TRITON_FALLBACK and not USE_TRITON:
            record(repo, "load", "RETRY", f"{cause}: {line}")
            USE_TRITON = True
            label += TRITON_LABEL
            continue
        if cause == "transformers too old" and ALT_TRANSFORMERS and "alt" not in tried:
            record(repo, "load", "RETRY", f"{cause} on SGLang's pinned transformers: {line}")
            tried.add("alt")
            py = alt_env()
            label += f" [non-default: transformers {ALT_TRANSFORMERS} in a second SGLang venv]"
            continue
        record(repo, "load", "FAIL", f"{cause}: {line}{label}")
        print("\n".join(text.splitlines()[-15:]))
        raise LoadFailed

def backend(log):
    text = open(log, encoding="utf-8", errors="ignore").read()
    return "SGLang's Transformers backend" if "Using Transformers backend" in text else "SGLang's native model class"

def alt_env():
    global ALT_PY
    if ALT_PY is None:
        print(f"  building a second SGLang venv with transformers {ALT_TRANSFORMERS} (~5 min) ...", flush=True)
        ALT_PY = build_sglang_env("/content/sglang-env-tf", extra=[f"transformers=={ALT_TRANSFORMERS}"])
        VERSIONS["alt_env"] = {"transformers": ALT_TRANSFORMERS,
                               "sglang_patch": patch_sglang(ALT_PY) if PATCH_SGLANG else "off"}
    return ALT_PY

def diagnose(repo):
    # same prompts, three paths: SGLang chat API, SGLang /generate with hf:'s token ids (server still up)
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(repo, token=os.environ.get("HF_TOKEN"))
    rows = []
    for code_, prompt, expect in PROBES:
        msgs = [{"role": "user", "content": prompt}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
        ids = ids["input_ids"] if hasattr(ids, "keys") else ids
        ids = ids[0] if ids and isinstance(ids[0], list) else list(ids)
        chat = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", timeout=120, json={
            "model": repo, "messages": msgs, "max_tokens": 32, "temperature": 0.0}).json()
        gen = requests.post(f"http://127.0.0.1:{PORT}/generate", timeout=120, json={
            "input_ids": ids, "sampling_params": {"temperature": 0.0, "max_new_tokens": 32}}).json()
        rows.append({"lang": code_, "expect": expect, "hf_prompt_tokens": len(ids),
                     "sglang_prompt_tokens": chat.get("usage", {}).get("prompt_tokens"),
                     "sglang_chat": (chat["choices"][0]["message"]["content"] or "").strip()[:60],
                     "sglang_same_ids": (gen.get("text") or "").strip()[:60]})
    return rows

def diagnose_hf(repo, rows):
    # the third path, hf: in this process; the server must be stopped first (GPU memory)
    import torch
    from auditkit.model.hf_gen import HFGenModel
    m = HFGenModel(repo, device="cuda", name=f"hf:{repo}", token=os.environ.get("HF_TOKEN"))
    for r, (code_, prompt, expect) in zip(rows, PROBES):
        g = m.generate([Request(prompt=prompt, params={"messages": [{"role": "user", "content": prompt}],
                                                        "max_tokens": 32, "temperature": 0.0})])[0].completions[0]
        r["hf"] = (g.text or "").strip()[:60]
    m.unload(); del m; torch.cuda.empty_cache()
    hit = lambda text, e: any(x in text for x in e)
    score = {k: sum(hit(r[k], r["expect"]) for r in rows) for k in ("sglang_chat", "sglang_same_ids", "hf")}
    extra_tokens = {r["lang"]: r["sglang_prompt_tokens"] - r["hf_prompt_tokens"] for r in rows
                    if r["sglang_prompt_tokens"] is not None}
    same = lambda a, b: all(r[a][:24] == r[b][:24] for r in rows)
    if any(extra_tokens.values()):
        verdict = f"SGLang's chat prompt differs from hf:'s by {extra_tokens} tokens: a prompt-building difference"
    elif same("sglang_same_ids", "hf") and not same("sglang_chat", "hf"):
        verdict = "same ids give hf:'s answers: SGLang's chat path builds a different prompt"
    elif same("sglang_chat", "sglang_same_ids") and not same("sglang_same_ids", "hf"):
        verdict = "identical prompts, different answers: a numerics difference between the runtimes, not a prompt bug"
    elif same("sglang_chat", "hf"):
        verdict = "all three agree: no difference on this GPU"
    else:
        verdict = "mixed: see the per-language rows"
    return score, verdict

def wait(proc, timeout_s):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            return False
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(10)
    return False

def stop(proc):
    try:
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM); proc.wait(timeout=120)
    except Exception:
        try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except Exception: pass
    for _ in range(30):                           # the next model needs the memory back
        if free_gb() > GPU_GB - 2: break
        time.sleep(2)

def delete_weights(repo):
    for r in scan_cache_dir().repos:
        if r.repo_id == repo:
            shutil.rmtree(r.repo_path, ignore_errors=True)
    # huggingface_hub's Xet downloads keep a chunk cache of their own, next to the model cache;
    # removing a model's weights leaves it, and on Colab's disk it fills up across nine models
    from huggingface_hub import constants
    shutil.rmtree(os.path.join(constants.HF_HOME, "xet"), ignore_errors=True)

def free_disk_gb():
    from huggingface_hub import constants
    os.makedirs(constants.HF_HUB_CACHE, exist_ok=True)
    return shutil.disk_usage(constants.HF_HUB_CACHE).free / 2**30

def probes(model):
    right, answers = 0, {}
    for code_, prompt, expect in PROBES:
        g = model.generate([Request(prompt=prompt, params={"messages": [{"role": "user", "content": prompt}],
                                                            "max_tokens": 32, "temperature": 0.0})])[0].completions[0]
        text = (f"ERR {g.error}" if g.finish_reason == "error" else
                (g.text or "").strip().replace("\n", " ") or f"EMPTY (finish={g.finish_reason})")[:70]
        answers[code_] = text
        right += any(e in text for e in expect)
    return right, answers

def image_check(model):
    from PIL import Image
    img = Image.new("RGB", (224, 224), "white")
    img.paste((220, 20, 20), (56, 56, 168, 168))
    q = "What colour is the square in the picture? Answer with one word."
    g = model.generate([Request(prompt=q, params={"messages": [{"role": "user", "content": q}], "images": [img],
                                                  "max_tokens": 16, "temperature": 0.0})])[0].completions[0]
    return (g.text or g.error or "").strip()[:80]

## Gate check (a few seconds)

All nine are gated. A licence you haven't accepted would cost a failed load mid-sweep, so the check runs first.

In [ ]:
GATED = []
for repo, *_ in MODELS:
    try:
        hf_hub_download(repo, "config.json", token=os.environ.get("HF_TOKEN"))
    except Exception as e:
        GATED.append(repo)
        print(f"  no access: {repo} ({type(e).__name__}) -> accept at https://huggingface.co/{repo}")
print("all nine accessible" if not GATED else
      f"{len(GATED)} not accessible: accept their licences with this token's account, then re-run this cell. "
      "They are recorded as SKIP otherwise.")

## The sweep

In [ ]:
import pandas as pd
class LoadFailed(Exception):
    pass
DIAG = {}
for repo, arch, need, is_vision in MODELS:
    print(f"\n=== {repo} ({arch}, needs ~{need} GB) ===")
    try:
        archs, found, model_type = registry(repo)
        record(repo, "registry", "declared" if found else "absent",
               f"model_type={model_type} architectures={archs} -> {found or 'no class in sglang/srt/models'}")
    except Exception as e:
        record(repo, "registry", "unknown", f"{type(e).__name__}: {e}")
    if repo in GATED:
        record(repo, "load", "SKIP", "licence not accepted for this HF token (see the gate check)"); continue
    extra, label = (), ""
    disk_need = need + 10                       # weights (~ the GPU need in bf16) plus working room
    if free_disk_gb() < disk_need:
        delete_weights(repo)                    # a half-finished download from an earlier try
        if free_disk_gb() < disk_need:
            record(repo, "load", "SKIP", f"needs ~{disk_need} GB of disk to download, {free_disk_gb():.0f} GB free"); continue
    if GPU_GB < need:
        if ALLOW_FP8_FOR_LARGE and GPU_GB >= need * 0.55:
            extra, label = ("--quantization", "fp8"), " [FP8: not comparable to the bf16 hf: column]"
        else:
            record(repo, "load", "SKIP", f"needs ~{need} GB in bf16, GPU has {GPU_GB:.0f} (H100 80 GB+ runs it)"); continue
    log = f"/tmp/sglang_{repo.split('/')[-1]}.log"
    proc, rows = None, None
    try:
        proc, t0, label = try_load(repo, log, extra, label)
        record(repo, "load", "PASS", f"healthy after {time.time()-t0:.0f}s via {backend(log)}{label}")
        m = APIModel(repo, api_base=f"http://127.0.0.1:{PORT}/v1", api_key="EMPTY", name=f"api:sglang/{repo}")
        right, answers = probes(m)
        record(repo, "text", "PASS" if right >= 4 else ("PARTIAL" if right else "FAIL"),
               f"{right}/{len(PROBES)} :: " + " | ".join(f"{k}={v!r}" for k, v in answers.items()))
        if is_vision:
            reply = image_check(m)
            record(repo, "image", "PASS" if "red" in reply.lower() else "FAIL", reply)
        if repo in DIAGNOSE:
            rows = diagnose(repo)
    except LoadFailed:
        pass                                      # recorded above; the weights are still deleted below
    except Exception as e:
        record(repo, "harness", "ERROR", f"{type(e).__name__}: {e}")
    finally:
        if proc is not None:
            stop(proc)
    if rows:
        try:
            score, verdict = diagnose_hf(repo, rows)
            record(repo, "diagnose", "INFO", f"{verdict}; correct: {score}")
            DIAG[repo] = rows
            display(pd.DataFrame(rows).drop(columns=["expect"]))
        except Exception as e:
            record(repo, "diagnose", "ERROR", f"{type(e).__name__}: {e}")
    if DELETE_AFTER:
        delete_weights(repo)

## Results

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS)
table = df.pivot_table(index="model", columns="check", values="outcome", aggfunc="last")   # last: a retry's outcome.reindex([m[0] for m in MODELS])
display(table[[c for c in ("registry", "load", "text", "image", "diagnose", "harness") if c in table]])
display(df[df.outcome.isin(["FAIL", "ERROR", "PARTIAL", "absent", "unknown"])][["model", "check", "outcome", "detail"]])
with open("sglang_cohere_matrix.json", "w") as f:
    json.dump({"versions": VERSIONS, "gpu": GPU_NAME, "gpu_gb": round(GPU_GB, 1),
               "date": time.strftime("%Y-%m-%d"), "results": RESULTS, "diagnose": DIAG}, f, indent=1, ensure_ascii=False)
print("wrote sglang_cohere_matrix.json:", VERSIONS, GPU_NAME)

### Reading it

- **registry `absent` + load PASS via Transformers backend:** SGLang has no native class, but it serves the model through its generic Transformers backend. That's the Aya Vision case, working with `PATCH_SGLANG`.
- **load PASS with `[non-default: transformers …]`:** it loaded only in the second venv. SGLang's own pin can't run it (North); report it that way.
- **diagnose:** the verdict line says whether a score gap against `hf:` comes from how SGLang builds the prompt or from the runtime's numerics.
- **registry `declared` + load FAIL:** the interesting case. The source knows the model, but it still doesn't run, and the `detail` column says why.
- **load FAIL `transformers too old`:** SGLang's transformers pin, not SGLang itself; `check_compat()` printed above warns about it.
- **text PARTIAL or FAIL on a model that loads:** a quality result, comparable with the `hf:` column, because it uses the same probes and the same rule.
- **Send back** `sglang_cohere_matrix.json` (in the Files pane) or the two tables above.